# word2vec as Implicit Matrix Factorization

Topic 6 built word vectors by hand: count co-occurrences, reweight by PMI, truncate an SVD. word2vec learns them by
stochastic gradient descent on a classification loss and never forms a matrix. This notebook shows that the two are
closer than they look, and where they part. The **objective** decides which matrix is fitted: shifted PMI, weighted by
counts and softly clipped. The **optimizer** decides which factorization of it we get, because the loss sees only the
product $WC^\top$ of word and context vectors.

Every number is computed by `word2vec_implicit_factorization.py`, which owns them. The text is Topic 6's: Topic 3's
training split of the shared finance corpus, with its L2 and L8 window co-occurrence matrices, and Topic 6's planted
sector benchmark (112 dev and 115 test nouns).

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "word2vec_implicit_factorization.py").exists()
                       else here / "notebooks" / "word2vec-implicit-factorization"))
import word2vec_implicit_factorization as w2v
ds = w2v.ds

for ctx in w2v.CONTEXTS:
    C = ds.cooc(ctx)
    print(f"{ctx}: N = {int(C.sum()):,} pairs over {C.nnz:,} observed (word, context) cells "
          f"({C.nnz / C.shape[0] ** 2:.2%} of {C.shape[0]:,}²)")

## 1. Three objectives, one matrix

Write $\#(w, c)$ for Topic 6's pair counts, $\#(w) = \sum_c \#(w, c)$, $N = \sum \#$, and
$P_\alpha(c) \propto \#(c)^\alpha$. Word and context vectors are the rows of $W, C \in \mathbb{R}^{V \times d}$, and the
score of a pair is $x_{wc} = \langle \mathbf{w}_w, \mathbf{c}_c \rangle$.

- **Def 1 (skip-gram, full softmax).** $L_{\text{soft}} = -\sum_{w,c} \#(w,c) \log \operatorname{softmax}(x_{w\cdot})_c$.
- **Def 2 (negative sampling).** $L_{\text{NS}} = -\sum_{w,c} \big[a \log\sigma(x_{wc}) + b \log\sigma(-x_{wc})\big]$ with
  $a = \#(w,c)$ and $b = k\,\#(w)\,P_\alpha(c)$, the expected number of times $(w, c)$ is drawn as a negative.
- **Def 3 (NCE).** The same loss with the logit $s = x - \log(k P_n(c))$.

**Prop 1.** The softmax optimum is $x = \log P(c \mid w) + \kappa_w$, attained only if every pair is observed.

**Thm 1 (Levy and Goldberg 2014).** $\varphi(x) = a \log\sigma(x) + b\log\sigma(-x)$ has $\varphi'' = -(a+b)\sigma(x)\sigma(-x) < 0$
and, for $a > 0$, the unique maximizer $x^* = \log(a/b) = \mathrm{PMI}_\alpha(w, c) - \log k$. For $a = 0$ it has none.

**Prop 2.** NCE's optimum is $\log P(c \mid w)$, and $x^*_{\text{NCE}} - x^*_{\text{NS}} = \log(k P_n(c))$, a column
offset. All three optima are PMI plus a row-constant and a column-constant matrix.

In [ ]:
w2v.test_three_optima()
w2v.test_sgns_gradient()
w2v.test_sampled_loss_is_unbiased()
w2v.test_corpus_optimum()
rng = np.random.default_rng(0)
counts = w2v._small_instance(rng, 3, 4) + 1
print("x* = PMI_0.75 − log 5 on a random 3 × 4 instance:\n", np.round(w2v.sgns_optimum_dense(counts, 0.75, 5), 3))
print("NCE − NS (one value per column):\n", np.round(w2v.nce_optimum(counts, 0.75, 5) - w2v.sgns_optimum_dense(counts, 0.75, 5), 3))

## 2. What SGNS minimizes at low rank

**Thm 2 (weighted logistic PCA).** With $y = a/(a+b)$,
$$L_{\text{NS}}(X) - \sum (a+b) H(y) = \sum_{w,c} (a+b)\, \mathrm{KL}\big(\mathrm{Bern}(y) \,\|\, \mathrm{Bern}(\sigma(x_{wc}))\big),$$
a Bregman divergence of the softplus. At $x^*$ the curvature is $ab/(a+b)$.

**Cor 1.** Near the optimum SGNS is least squares weighted by $ab/(a+b) \le \min(a, b)$: rare pairs get little
weight. Unobserved cells are pushed down by $b \log(1 + e^x)$ instead of being clipped to 0. Truncated SVD of
$\mathrm{SPPMI}_k$ weights every entry alike.

In [ ]:
w2v.test_logistic_pca_identity()
w2v.test_fit_by_count_band()
f = w2v.fit_summary()
print("controlled model (L8, d = 25, k = 5, α = 0.75, 40 epochs): w·c against PMI_α − log k by pair count")
for name, n, r, mae in zip(w2v.BAND_NAMES, f["n"], f["r"], f["mae"]):
    print(f"  seen {name:>6}: {n:>6} pairs   r = {r:.3f}   mean |error| = {mae:.3f}")
print(f"mean w·c: observed {f['x_obs_mean']:.2f}, unobserved {f['x_unobs_mean']:.2f}")
print(f"loss per pair {f['loss']:.4f} vs Thm 2's floor {f['floor']:.4f}; gradient norm {f['grad_norm']:.4f}")
print("the bias direction:", {k: round(v, 3) for k, v in w2v.bias_direction().items()})

## 3. The algorithm

**Def 4 (word2vec's SGD).** Each epoch is a fresh permutation of the $N$ pairs in minibatches of 512. Every pair gets
$k$ negatives drawn from $P_\alpha$, the learning rate decays linearly from $\eta_0 = 0.05$, and the initialization is the
word2vec C code's, $W \sim U[-\tfrac{1}{2d}, \tfrac{1}{2d}]$, $C = 0$. The sampled loss is unbiased for $L_{\text{NS}}$
(checked above). The controlled run trains 40 epochs and the grid 20.

In [ ]:
for h in w2v.train_sgns(w2v.CONTROLLED).history:
    extra = (f"  r(seen once) {h['band_r'][0]:.3f}  r(100+) {h['band_r'][-1]:.3f}  gauge σ ∈ "
             f"[{h['gauge_sv'][0]:.3f}, {h['gauge_sv'][1]:.3f}]") if "band_r" in h else ""
    print(f"epoch {h['epoch']:>2}: loss {h['loss']:.4f}  imbalance {h['imbalance']:.3f}{extra}")

## 4. Gauge and balance

**Prop 3.** Every rank-$d$ factorization of $X = WC^\top$ is $(WA, CA^{-\top})$ with $A \in GL(d)$, so the loss fixes the
word Gram matrix only up to $WMW^\top$, $M \succ 0$.

**Thm 3.** Along gradient flow $D = W^\top W - C^\top C$ is constant. One step of gradient descent changes it by
exactly $\eta^2(\nabla_W^\top \nabla_W - \nabla_C^\top \nabla_C)$.

**Lemma 1.** Balanced factors ($W^\top W = C^\top C$) of $U\Sigma V^\top$ are $U\Sigma^{1/2}R$ and $V\Sigma^{1/2}R$.

**Cor 2.** Balanced word vectors are Topic 6's $p = \tfrac12$ for the fitted matrix, and for a symmetric product
$\tfrac14 (W + C)(W + C)^\top = [X]_+$, Topic 6's Corollary 1.

In [ ]:
w2v.test_gauge_invariance()
for name, cell in w2v.gauge_cells().items():
    print(f"{name:>8}: test AUC {cell['test_auc']}, 1-NN {cell['test_nn']};  strong →",
          [n for n, _ in cell["neighbors"]["strong"]])

In [ ]:
w2v.test_gradient_flow_conservation()
w2v.test_balanced_factorization()
w2v.test_corpus_balance()
print("drift at a fixed horizon, halving η:", {k: round(v, 4) for k, v in w2v.drift_summary().items()})
print("symmetric instance, balanced start:", {k: float(f"{v:.3g}") for k, v in w2v.symmetric_dynamics().items()})
print("corpus:", {k: float(f"{v:.4g}") for k, v in w2v.balance_summary().items()})

## 5. Recovering the planted sectors

Topic 6's protocol: pair AUC and 1-NN on the 115 test nouns, configurations chosen on the 112 dev nouns. The
controlled pair puts SGNS and Topic 6's selected SVD on the same matrix (L8, $\mathrm{SPPMI}_5$, $\alpha = 0.75$,
$d = 25$).

In [ ]:
w2v.test_recovery_on_test_words()
c = w2v.sgns_comparison()
print(f"controlled: SGNS test AUC {c['ctl_test_auc']:.3f}, 1-NN {c['ctl_test_nn']:.3f};  "
      f"SVD {c['ctl_svd_test_auc']:.3f}, {c['ctl_svd_test_nn']:.3f}")
print(f"  AUC difference {c['ctl_vs_svd_auc_diff']:+.3f} (jackknife se {c['ctl_vs_svd_auc_diff_se']:.3f})")
for g in w2v.GRID:
    s = w2v.sgns_scores(g)
    print(f"grid {g.context}, d = {g.d:>3}: dev AUC {s['dev_auc']:.3f}  test AUC {s['test_auc']:.3f}  test 1-NN {s['test_nn']:.3f}")
print("selected on dev:", c["sel"])
print(f"rare pairs (seen ≤ 4) in the benchmark rows: {c['mech_rare_share_support']:.1%} of the support, "
      f"{c['mech_rare_share_sppmi_mass']:.1%} of SPPMI₅'s squared mass, {c['mech_rare_share_sgns_weight']:.1%} of SGNS's weight")

## 6. Finance case study: polarity isn't a direction

A leave-one-out difference-of-means probe on the stand-in lexicon's 35 polarity words (18 positive, 17 negative).
Topic 6 found that windows carry no polarity evidence (L2) or backwards evidence (L8). A factorization of those counts
has nothing more to find.

In [ ]:
w2v.test_finance_findings()
fin = w2v.finance_summary()
for ctx in w2v.CONTEXTS:
    print(ctx, {m: round(fin[f"probe_{m}_{ctx}"], 3) for m in ("sgns", "sgns_wc", "svd", "ppmi")},
          f"Topic 6 SO {fin[f'so_acc_{ctx}']:.3f}")
print(f"chance 0.5 ± {fin['chance_se']:.3f}; the best probe is {fin['probe_max_z']:.2f} standard errors above chance")
print("strong's SGNS neighbors at L2:", [(w, round(s, 3)) for w, s in fin["strong_neighbors_L2"]])

## The lab's numbers, and the full harness

`viz_payload()` holds every number the lab shows. Panel A has *revenue*'s most associated context in each count band
at L8, and one context it never meets. Panel B has the controlled model's fit on 200 sampled pairs at each
checkpoint, and the gauge variants.

In [ ]:
w2v.test_viz_payload()
P = w2v.viz_payload()
for cell in P["pairs"]:
    print(f"revenue·{cell['context']:<12} a = {cell['a']:>5.0f}   x* = {cell['x_star']['0.75|5']}   "
          f"weight = {cell['weight']['0.75|5']}   trained w·c = {cell['trained']}")

In [ ]:
assert w2v._run_harness() == 0